# HW2: Springy Double Pendulum (Lagrangian dynamics)


  $$M(q)\ddot q + C(q,\dot q)\dot q + g(q) = Q.$$
- **Task 2 **: RK4 simulation with **$Q=0$**, plus trajectory/energy plots and animation.
- **Task 3 **: RK4 simulation with **$Q_1(t)=10(\sin t-q_1)-0.2\dot q_1$**, plus the same outputs.

All figures/animations will be saved in `HW2/outputs/`.

Generalized coordinates used (to match $r_2(0)=[1.8,0]$):

$$q = [q_1,\; q_2,\; \ell]^T$$

where $q_1$ is the first link absolute angle, $q_2$ is the spring absolute angle, and $\ell$ is the instantaneous spring length.


In [13]:
from pathlib import Path

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib import animation

_cwd = Path.cwd().resolve()
if (_cwd / 'HW2').is_dir():
    _root = _cwd
elif _cwd.name.lower() == 'hw2' and (_cwd.parent / 'HW2').is_dir():
    _root = _cwd.parent
else:
    _root = _cwd

OUTPUT_DIR = _root / 'HW2' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Make MP4 export work even if ffmpeg isn't installed system-wide
try:
    import imageio_ffmpeg as _imageio_ffmpeg

    plt.rcParams['animation.ffmpeg_path'] = _imageio_ffmpeg.get_ffmpeg_exe()
except Exception:
    pass

m1 = 0.5
m2 = 1.0
l1 = 1.0
l2 = 0.8
k = 200.0
g = 9.81

print('Outputs will be saved to:', OUTPUT_DIR)


Outputs will be saved to: /Users/anashamrouni/Documents/InnopolisUniversity/Bachelor/RoboticSystems/HW2/outputs


In [14]:
# Symbolic derivation of M(q), C(q,qd), g(q)

# Generalized coordinates
q1, q2, ell = sp.symbols('q1 q2 ell', real=True)
qd1, qd2, elld = sp.symbols('qd1 qd2 elld', real=True)
qdd1, qdd2, elldd = sp.symbols('qdd1 qdd2 elldd', real=True)

q = sp.Matrix([q1, q2, ell])
qd = sp.Matrix([qd1, qd2, elld])
qdd = sp.Matrix([qdd1, qdd2, elldd])

# Position vectors (inertial frame x-right, y-up; gravity acts in -y)
r1 = sp.Matrix([l1*sp.cos(q1), l1*sp.sin(q1)])
e2 = sp.Matrix([sp.cos(q2), sp.sin(q2)])
r2 = r1 + ell*e2

# Jacobians for velocities: rdot = J(q) * qdot
J1 = r1.jacobian(q)
J2 = r2.jacobian(q)

v1 = J1 * qd
v2 = J2 * qd

T = sp.Rational(1,2)*m1*(v1.dot(v1)) + sp.Rational(1,2)*m2*(v2.dot(v2))
V = m1*g*r1[1] + m2*g*r2[1] + sp.Rational(1,2)*k*(ell - l2)**2

# Mass matrix from kinetic energy: T = 1/2 qd^T M qd
M = sp.hessian(T, qd)

# Gravity + spring generalized forces
G = sp.Matrix([sp.diff(V, qi) for qi in q])

# Christoffel symbols -> C(q,qd) matrix such that C(q,qd)*qd reproduces Coriolis/centrifugal terms
n = 3
Gamma = [[[
    sp.simplify(sp.Rational(1,2)*(sp.diff(M[i,j], q[k]) + sp.diff(M[i,k], q[j]) - sp.diff(M[j,k], q[i])))
    for k in range(n)
] for j in range(n)] for i in range(n)]

C = sp.zeros(n, n)
for i in range(n):
    for j in range(n):
        C[i, j] = sp.simplify(sum(Gamma[i][j][k]*qd[k] for k in range(n)))

# Final symbolic expression: M*qdd + C*qd + G = Q
print('M(q) shape:', M.shape)
print('C(q,qd) shape:', C.shape)
print('G(q) shape:', G.shape)


M(q) shape: (3, 3)
C(q,qd) shape: (3, 3)
G(q) shape: (3, 1)


## Task 1 — Derive dynamics in manipulator form

- $M(q)$ from the quadratic form of $T(q,\dot q)$
- $C(q,\dot q)$ using Christoffel symbols
- $g(q)=\nabla_q V(q)$ (gravity + spring)

Then convert them to numerical functions and use them in simulation.


In [15]:
# Numerical functions (sympy -> numpy)

M_func = sp.lambdify((q1, q2, ell), M, 'numpy')
C_func = sp.lambdify((q1, q2, ell, qd1, qd2, elld), C, 'numpy')
G_func = sp.lambdify((q1, q2, ell), G, 'numpy')

r1_func = sp.lambdify((q1,), r1, 'numpy')
r2_func = sp.lambdify((q1, q2, ell), r2, 'numpy')

T_func = sp.lambdify((q1, q2, ell, qd1, qd2, elld), T, 'numpy')
V_func = sp.lambdify((q1, q2, ell), V, 'numpy')


def dynamics_rhs(t, x, torque_mode='zero'):
    """x = [q1,q2,ell, qd1,qd2,elld]. Returns xdot."""
    q1v, q2v, ellv, qd1v, qd2v, elldv = x

    Mv = np.array(M_func(q1v, q2v, ellv), dtype=float)
    Cv = np.array(C_func(q1v, q2v, ellv, qd1v, qd2v, elldv), dtype=float)
    Gv = np.array(G_func(q1v, q2v, ellv), dtype=float).reshape(3)

    if torque_mode == 'zero':
        tau1 = 0.0
    elif torque_mode == 'q1_torque':
        # Task 3 input torque on joint 1
        tau1 = 10.0 * (np.sin(t) - q1v) - 0.2 * qd1v
    else:
        raise ValueError('Unknown torque_mode')

    Q = np.array([tau1, 0.0, 0.0], dtype=float)

    qdv = np.array([qd1v, qd2v, elldv], dtype=float)
    qddv = np.linalg.solve(Mv, Q - (Cv @ qdv + Gv))

    return np.array([qd1v, qd2v, elldv, qddv[0], qddv[1], qddv[2]], dtype=float)


def rk4_integrate(f, t0, tf, dt, x0, **kwargs):
    ts = np.arange(t0, tf + 1e-12, dt)
    xs = np.zeros((len(ts), len(x0)), dtype=float)
    xs[0] = x0

    for i in range(len(ts) - 1):
        t = ts[i]
        x = xs[i]
        k1 = f(t, x, **kwargs)
        k2 = f(t + dt / 2, x + dt * k1 / 2, **kwargs)
        k3 = f(t + dt / 2, x + dt * k2 / 2, **kwargs)
        k4 = f(t + dt, x + dt * k3, **kwargs)
        xs[i + 1] = x + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6

        # Clamp spring length to remain positive
        xs[i + 1, 2] = max(xs[i + 1, 2], 1e-3)

    return ts, xs


In [16]:
def compute_energy(xs):
    q1v = xs[:, 0]
    q2v = xs[:, 1]
    ellv = xs[:, 2]
    qd1v = xs[:, 3]
    qd2v = xs[:, 4]
    elldv = xs[:, 5]

    Tvals = np.array([T_func(q1v[i], q2v[i], ellv[i], qd1v[i], qd2v[i], elldv[i]) for i in range(len(xs))], dtype=float)
    Vvals = np.array([V_func(q1v[i], q2v[i], ellv[i]) for i in range(len(xs))], dtype=float)
    return Tvals, Vvals, Tvals + Vvals


def compute_positions(xs):
    r1s = np.zeros((len(xs), 2), dtype=float)
    r2s = np.zeros((len(xs), 2), dtype=float)
    for i, (q1v, q2v, ellv) in enumerate(xs[:, :3]):
        r1s[i] = np.array(r1_func(q1v), dtype=float).reshape(2)
        r2s[i] = np.array(r2_func(q1v, q2v, ellv), dtype=float).reshape(2)
    return r1s, r2s


## Tasks 2–3 — RK4 simulation, plots, and animation

We integrate the state

$$x=[q_1,\;q_2,\;\ell,\;\dot q_1,\;\dot q_2,\;\dot\ell]^T$$

with a fixed-step **RK4** scheme.

- **Task 2**: set $Q=0$.
- **Task 3**: set $Q_1(t)=10(\sin t-q_1)-0.2\dot q_1$, and $Q_2=Q_3=0$.

For each case`:
- trajectory plot
- energy plot ($T$, $V$, $T+V$)
- animation


In [17]:
# Simulation settings

t0, tf = 0.0, 10.0
# Small dt due to the stiff spring (k=200 N/m)
dt = 0.002

# Initial condition: r2(0) = [1.8, 0], system initially at rest
# With q1(0)=0, q2(0)=0, ell(0)=l2 => r2(0) = [l1+l2, 0] = [1.8, 0]
x0 = np.array([0.0, 0.0, l2, 0.0, 0.0, 0.0], dtype=float)


In [18]:
def run_case(case_name, torque_mode):
    print(f'Running: {case_name} ({torque_mode})')
    ts, xs = rk4_integrate(dynamics_rhs, t0, tf, dt, x0, torque_mode=torque_mode)

    r1s, r2s = compute_positions(xs)
    Tvals, Vvals, Evals = compute_energy(xs)

    # Trajectory plot
    plt.figure(figsize=(6, 4))
    plt.plot(r2s[:, 0], r2s[:, 1], label='$m_2$ path')
    plt.plot(r1s[:, 0], r1s[:, 1], label='$m_1$ path', alpha=0.7)
    plt.gca().set_aspect('equal', 'box')
    plt.grid(True, alpha=0.3)
    plt.xlabel('x [m]')
    plt.ylabel('y [m]')
    plt.title(f'Trajectories: {case_name}')
    plt.legend()
    traj_path = OUTPUT_DIR / f'{case_name}_trajectories.png'
    plt.tight_layout()
    plt.savefig(traj_path, dpi=200)
    plt.close()

    # Energy plot
    plt.figure(figsize=(6, 4))
    plt.plot(ts, Tvals, label='T')
    plt.plot(ts, Vvals, label='V')
    plt.plot(ts, Evals, label='E = T + V', linewidth=2)
    plt.grid(True, alpha=0.3)
    plt.xlabel('t [s]')
    plt.ylabel('Energy [J]')
    plt.title(f'Energy: {case_name}')
    plt.legend()
    energy_path = OUTPUT_DIR / f'{case_name}_energy.png'
    plt.tight_layout()
    plt.savefig(energy_path, dpi=200)
    plt.close()

    # Animation (MP4)
    stride = max(1, int(0.02 / dt))
    ts_a = ts[::stride]
    r1_a = r1s[::stride]
    r2_a = r2s[::stride]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.set_aspect('equal', 'box')
    ax.grid(True, alpha=0.3)

    all_xy = np.vstack([np.zeros((len(r2_a), 2)), r1_a, r2_a])
    pad = 0.2
    ax.set_xlim(all_xy[:, 0].min() - pad, all_xy[:, 0].max() + pad)
    ax.set_ylim(all_xy[:, 1].min() - pad, all_xy[:, 1].max() + pad)

    link_line, = ax.plot([], [], '-k', lw=2)
    spring_line, = ax.plot([], [], '-C1', lw=2)
    m1_pt, = ax.plot([], [], 'o', color='C0', ms=8)
    m2_pt, = ax.plot([], [], 'o', color='C3', ms=10)
    time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)

    def init():
        link_line.set_data([], [])
        spring_line.set_data([], [])
        m1_pt.set_data([], [])
        m2_pt.set_data([], [])
        time_text.set_text('')
        return link_line, spring_line, m1_pt, m2_pt, time_text

    def update(i):
        r1i = r1_a[i]
        r2i = r2_a[i]
        link_line.set_data([0, r1i[0]], [0, r1i[1]])
        spring_line.set_data([r1i[0], r2i[0]], [r1i[1], r2i[1]])
        m1_pt.set_data([r1i[0]], [r1i[1]])
        m2_pt.set_data([r2i[0]], [r2i[1]])
        time_text.set_text(f't = {ts_a[i]:.2f} s')
        return link_line, spring_line, m1_pt, m2_pt, time_text

    ani = animation.FuncAnimation(fig, update, frames=len(ts_a), init_func=init, blit=True, interval=20)

    mp4_path = OUTPUT_DIR / f'{case_name}_animation.mp4'
    writer = animation.FFMpegWriter(fps=30, codec='h264', bitrate=1800)
    ani.save(mp4_path, writer=writer)
    plt.close(fig)

    print('Saved:', traj_path.name, energy_path.name, mp4_path.name)
    return ts, xs


ts0, xs0 = run_case('case2_Q0', torque_mode='zero')
ts1, xs1 = run_case('case3_torque', torque_mode='q1_torque')


Running: case2_Q0 (zero)
Saved: case2_Q0_trajectories.png case2_Q0_energy.png case2_Q0_animation.mp4
Running: case3_torque (q1_torque)
Saved: case3_torque_trajectories.png case3_torque_energy.png case3_torque_animation.mp4
